# DS 110 SBA Case: Logit Model Homework
## Austin Melendez
### 5/6/2026


In [18]:
!pip install dmba

In [85]:
import pandas as pd
import numpy as np

import statsmodels.api as sm

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix

import matplotlib.pyplot as plt

from dmba import classificationSummary

In [20]:
sba = pd.read_csv('./data/SBAcase.11.13.17.csv')

sba.head()

,Selected,LoanNr_ChkDgt,Name,City,State,Zip,Bank,BankState,NAICS,ApprovalDate,...,ChgOffPrinGr,GrAppv,SBA_Appv,New,RealEstate,Portion,Recession,daysterm,xx,Default
0,0,1004285007,SIMPLEX OFFICE SOLUTIONS,ANAHEIM,CA,92801,CALIFORNIA BANK & TRUST,CA,532420,15074,...,0,30000,15000,0,0,0.5,0,1080,16175.0,0
1,1,1004535010,DREAM HOME REALTY,TORRANCE,CA,90505,CALIFORNIA BANK & TRUST,CA,531210,15130,...,0,30000,15000,0,0,0.5,1,1680,17658.0,0
2,0,1005005006,"Winset, Inc. dba Bankers Hill",SAN DIEGO,CA,92103,CALIFORNIA BANK & TRUST,CA,531210,15188,...,0,30000,15000,0,0,0.5,0,1080,16298.0,0
3,1,1005535001,Shiva Management,SAN DIEGO,CA,92108,CALIFORNIA BANK & TRUST,CA,531312,15719,...,0,50000,25000,0,0,0.5,0,1080,16816.0,0
4,1,1005996006,"GOLD CROWN HOME LOANS, INC",LOS ANGELES,CA,91345,SBA - EDF ENFORCEMENT ACTION,CO,531390,16840,...,0,343000,343000,0,1,1.0,0,7200,24103.0,0


We can see we are correctly importing the data.

In [21]:
sba.head()

,Selected,LoanNr_ChkDgt,Name,City,State,Zip,Bank,BankState,NAICS,ApprovalDate,...,ChgOffPrinGr,GrAppv,SBA_Appv,New,RealEstate,Portion,Recession,daysterm,xx,Default
0,0,1004285007,SIMPLEX OFFICE SOLUTIONS,ANAHEIM,CA,92801,CALIFORNIA BANK & TRUST,CA,532420,15074,...,0,30000,15000,0,0,0.5,0,1080,16175.0,0
1,1,1004535010,DREAM HOME REALTY,TORRANCE,CA,90505,CALIFORNIA BANK & TRUST,CA,531210,15130,...,0,30000,15000,0,0,0.5,1,1680,17658.0,0
2,0,1005005006,"Winset, Inc. dba Bankers Hill",SAN DIEGO,CA,92103,CALIFORNIA BANK & TRUST,CA,531210,15188,...,0,30000,15000,0,0,0.5,0,1080,16298.0,0
3,1,1005535001,Shiva Management,SAN DIEGO,CA,92108,CALIFORNIA BANK & TRUST,CA,531312,15719,...,0,50000,25000,0,0,0.5,0,1080,16816.0,0
4,1,1005996006,"GOLD CROWN HOME LOANS, INC",LOS ANGELES,CA,91345,SBA - EDF ENFORCEMENT ACTION,CO,531390,16840,...,0,343000,343000,0,1,1.0,0,7200,24103.0,0


Next we want to partition the data into train and test splits. Based on the documentation we see the `Selected` variable identifies if it is training or validation data. 
- 1 = train
- 0 = valid

In [22]:
trainData = sba[sba.Selected == 1]
validData = sba[sba.Selected == 0]

print(trainData.shape)
print(validData.shape)

(1051, 35)
(1051, 35)


Looks like they used a 50/50 split for train/valid data.

# (a)
### 1. Statsmodels

The model in Table 7(a) uses:
- `New`
- `RealEstate`
- `DisbursementGross`
- `Portion`
- `Recession`

In [23]:
predictors = ['New', 'RealEstate', 'DisbursementGross', 'Portion', 'Recession']

X_train = trainData[predictors]
y_train = trainData['Default']

X_valid = validData[predictors]
y_valid = validData['Default']

In [24]:
X_train_sm = sm.add_constant(X_train)

logit_reg = sm.Logit(y_train, X_train_sm)

logit_result = logit_reg.fit()

print(logit_result.summary())

Optimization terminated successfully.
         Current function value: 0.514605
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:                Default   No. Observations:                 1051
Model:                          Logit   Df Residuals:                     1045
Method:                           MLE   Df Model:                            5
Date:                Wed, 06 May 2026   Pseudo R-squ.:                  0.1740
Time:                        23:52:16   Log-Likelihood:                -540.85
converged:                       True   LL-Null:                       -654.77
Covariance Type:            nonrobust   LLR p-value:                 3.112e-47
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
const                 1.3537      0.323      4.192      0.000       0.721       1.987
New     

In [25]:
### 2. sklearn lblinear solver

In [26]:
logit_liblinear = LogisticRegression(
    solver='liblinear',
    random_state=1
)

logit_liblinear.fit(X_train, y_train)

print('Intercept')
print(np.round(logit_liblinear.intercept_, 4))

print('\nCoefficients')

for var, coef in zip(predictors, logit_liblinear.coef_[0]):
    print(var, round(coef, 4))

Intercept
[-0.]

Coefficients
New -0.0
RealEstate -0.0
DisbursementGross -0.0
Portion -0.0
Recession 0.0


### 3. sklearn lbfgs solver

In [27]:
logit_lbfgs = LogisticRegression(
    solver='lbfgs',
    random_state=1,
    max_iter=1000
)

logit_lbfgs.fit(X_train, y_train)

print('Intercept')
print(np.round(logit_lbfgs.intercept_, 4))

print('\nCoefficients')

for var, coef in zip(predictors, logit_lbfgs.coef_[0]):
    print(var, round(coef, 4))

Intercept
[1.0382]

Coefficients
New -0.1009
RealEstate -1.9317
DisbursementGross -0.0
Portion -2.2603
Recession 0.4807


The model in Table 8 uses:
- `RealEstate`
- `Portion`
- `Recession`

In [28]:
predictors = ['RealEstate', 'Portion', 'Recession']

X_train = trainData[predictors]
y_train = trainData['Default']

X_valid = validData[predictors]
y_valid = validData['Default']

In [29]:
### 1. Statsmodels

In [30]:
X_train_sm = sm.add_constant(X_train)

logit_reg = sm.Logit(y_train, X_train_sm)

logit_result = logit_reg.fit()

print(logit_result.summary())

Optimization terminated successfully.
         Current function value: 0.515108
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:                Default   No. Observations:                 1051
Model:                          Logit   Df Residuals:                     1047
Method:                           MLE   Df Model:                            3
Date:                Wed, 06 May 2026   Pseudo R-squ.:                  0.1732
Time:                        23:52:17   Log-Likelihood:                -541.38
converged:                       True   LL-Null:                       -654.77
Covariance Type:            nonrobust   LLR p-value:                 6.874e-49
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          1.3931      0.322      4.332      0.000       0.763       2.023
RealEstate    -2.1282      0.

### 2. sklearn liblinear solver

In [31]:
logit_liblinear = LogisticRegression(
    solver='liblinear',
    random_state=1
)

logit_liblinear.fit(X_train, y_train)

print('Intercept')
print(np.round(logit_liblinear.intercept_, 4))

print('\nCoefficients')

for var, coef in zip(predictors, logit_liblinear.coef_[0]):
    print(var, round(coef, 4))

Intercept
[0.9892]

Coefficients
RealEstate -2.096
Portion -2.3194
Recession 0.4977


### 3. sklearn lbfgs

In [32]:
logit_lbfgs = LogisticRegression(
    solver='lbfgs',
    random_state=1,
    max_iter=1000
)

logit_lbfgs.fit(X_train, y_train)

print('Intercept')
print(np.round(logit_lbfgs.intercept_, 4))

print('\nCoefficients')

for var, coef in zip(predictors, logit_lbfgs.coef_[0]):
    print(var, round(coef, 4))

Intercept
[1.0712]

Coefficients
RealEstate -2.0739
Portion -2.4487
Recession 0.4892


# (b) Estimated Logistic Regression Equations

Let:
- $p =$ probability of defualt

Then our fitted coefficients are:
- $\beta_0 =$ intercept
- $\beta_1 =$ `RealEstate` coefficient
- $\beta_2 =$ `Portion` coefficient
- $\beta_3 =$ `Recession` coefficient

In [88]:
b0 = logit_result.params['const']
b1 = logit_result.params['RealEstate']
b2 = logit_result.params['Portion']
b3 = logit_result.params['Recession']

print(round(b0,4),round(b1,4),round(b2,4),round(b3,4))

1.3931 -2.1282 -2.9875 0.5041


### (i) Logit Function

$$
\log\left(\frac{p}{1-p}\right)
=
1.3931
+ (-2.1282)(\text{RealEstate})
+ (-2.9875)(\text{Portion})
+ (0.5041)(\text{Recession})
$$

### (ii) Odds Function

$$
\frac{p}{1-p}
=
e^{
1.3931
+ (-2.1282)(\text{RealEstate})
+ (-2.9875)(\text{Portion})
+ (0.5041)(\text{Recession})
}
$$

### (iii) Probability Function

$$
p
=
\frac{1}
{
1 + e^{-(
1.3931
+ (-2.1282)(\text{RealEstate})
+ (-2.9875)(\text{Portion})
+ (0.5041)(\text{Recession})
)}
}
$$

# (c) Why where these variable selected?

The variables in Table 8 were selected because their p-values in Table 7(a) were statistically significant.

Small p-values indicate evidence that the variable contributes meaningful information for predicting loan default.
The selected predictors:
- RealEstate
- Portion
- Recession

were kept because their coefficients were significantly different from zero and improved predictive accuracy.

Variables with insignificant p-values were removed from the final model.

# (d) Interpretation of Coefficients

In [37]:
odds_ratios = np.exp(logit_result.params)

print(odds_ratios.round(4))

const         4.0272
RealEstate    0.1190
Portion       0.0504
Recession     1.6555
dtype: float64


### (i) Real Estate Loans
The coefficient for `RealEstate` is negative.
The odds ratio is:
$$ \mathrm{e}^{b_1} $$ 

In [38]:
print(np.exp(b1))

0.11904937261110025


The odds of default are only 12% as large as loans not backed by real estate.
    
So we therefore can conclude that loans backed by real estate are less likely to default.

### (ii) Recession
The coefficient for `Recession` is positive.
The odds ratio is:
$$ \mathrm{e}^{b_3} $$ 

Therefore loans active during a recession are more likely to default.

### (iii) Portion Guaranteed by SBA
The coefficient for `Portion` is positive.
The odds ratio is:
$$ \mathrm{e}^{b_3} $$ 

As the SBA guaranteed portion increases, the likelihood and odds of default increase.

# (e) Gains and Lift Charts

In [39]:
X_valid_sm = sm.add_constant(X_valid)

validProb = logit_result.predict(X_valid_sm)

In [83]:
gainData = pd.DataFrame({
    'actual': y_valid,
    'prob': validProb
})

gainData = gainData.sort_values(by='prob', ascending=False)

gainData['decile'] = pd.qcut(
    gainData.index,
    10,
    labels=False
)

overallRate = gainData.actual.mean()

liftTable = gainData.groupby('decile').actual.mean() / overallRate

print(liftTable)

decile
0    1.284773
1    1.550771
2    1.832730
3    1.860926
4    1.127834
5    0.338350
6    0.310154
7    0.563917
8    0.507525
9    0.620309
Name: actual, dtype: float64


### Gain Chart

In [86]:
cumulative = gainData.actual.cumsum() / gainData.actual.sum()

plt.figure(figsize=(8,5))

plt.plot(
    np.arange(1, len(cumulative)+1) / len(cumulative),
    cumulative
)

plt.xlabel('Proportion of Validation Data')
plt.ylabel('Cumulative Defaults Captured')
plt.title('Gains Chart')

plt.grid(True)

plt.show()

### Lift Chart

In [87]:
plt.figure(figsize=(8,5))

plt.plot(
    liftTable.index + 1,
    liftTable.values
)

plt.xlabel('Decile')
plt.ylabel('Lift')
plt.title('Lift Chart')

plt.grid(True)

plt.show()

### First Decile Lift

In [43]:
print(liftTable.iloc[0])

1.2847727876694128


The first decile lift measures how much better the model performs than random selection in identifying defaults among the highest-risk loans.

A lift greater than 1 indicates improved targeting performance.

# (f) Predict Carmichael Realty and SV Consulting
### Loan 1 — Carmichael Realty

In [67]:
loan1 = pd.DataFrame({
    'RealEstate': [1],
    'Portion': [0.75],
    'Recession': [0]
})

### Loan 2 - SV Consulting

In [68]:
loan2 = pd.DataFrame({
    'RealEstate': [0],
    'Portion': [0.40],
    'Recession': [0]
})

### Predict Probabilities

In [69]:
loan1_sm = sm.add_constant(loan1, has_constant='add')
loan2_sm = sm.add_constant(loan2, has_constant='add')

loan1Prob = logit_result.predict(loan1_sm)[0]
loan2Prob = logit_result.predict(loan2_sm)[0]

print('Loan 1 Probability:', round(loan1Prob, 4))
print('Loan 2 Probability:', round(loan2Prob, 4))

Loan 1 Probability: 0.0485
Loan 2 Probability: 0.5494


My values are the same as what they report in the report.

We can do classification using cutoff = 0.5

In [80]:
loan1Class = (
    'Higher Risk (Deny)'
    if loan1Prob >= 0.5
    else 'Lower Risk (Approve)'
)

loan2Class = (
    'Higher Risk (Deny)'
    if loan2Prob >= 0.5
    else 'Lower Risk (Approve)'
)

print('Loan 1:', loan1Class)
print('Loan 2:', loan2Class)

Loan 1: Lower Risk (Approve)
Loan 2: Higher Risk (Deny)


# (g) Classification Metrics
### Predicted Classes

In [71]:
validPred = [1 if p >= 0.5 else 0 for p in validProb]

### Classification Summary

In [72]:
classificationSummary(y_valid, validPred)

Confusion Matrix (Accuracy 0.6784)

       Prediction
Actual   0   1
     0 682  14
     1 324  31


### Confusion Matrix

In [73]:
cm = confusion_matrix(y_valid, validPred)

print(cm)

[[682  14]
 [324  31]]


### TN, FP, FN, TP

In [79]:
TN, FP, FN, TP = cm.ravel()

print('\nTN, FP, FN, TP')
print(TN, FP, FN, TP)


TN, FP, FN, TP
682 14 324 31


### Recall (Sensitivity)
$$
Recall
=
\frac{TP}{TP + FN}
$$

In [75]:
recall = TP / (TP + FN)

print(recall)

0.08732394366197183


### Specificity
$$
Specificity
=
\frac{TN}{TN + FP}
$$

In [76]:
specificity = TN / (TN + FP)

print(specificity)

0.9798850574712644


### Precision
$$
Precision
=
\frac{TP}{TP + FP}
$$

In [77]:
precision = TP / (TP + FP)

print(precision)

0.6888888888888889


### F1 Score
$$
F1
=
\frac{
2(Precision)(Recall)
}{
Precision + Recall
}
$$

In [78]:
f1 = 2 * (precision * recall) / (precision + recall)

print(f1)

0.155
